# Everything together

One question, asked of 46 different pieces of work:

> **Does this job need distributing?**

Each operation ran twice over the same files, once in a single process and once through a
distributed engine running locally.

The `Ratio` column is the gap between the two. It is not labelled as the cost of distribution,
because two things differ at once here: Spark distributes the work, and it is also a different
engine, a JVM runtime with a scheduler against a vectorized C++ engine running in-process. This
setup cannot say how much of the gap is which.

What it does show is that on one machine the distributed path paid the full price of
distributing and collected none of the benefit, which is the position of any job that never
needed more than one machine.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from bench import config as C, harness, report
import pandas as pd

# Reads whatever JSON is in results/. If cases have been added or renamed since
# the last run, clear that folder first or the totals will mix two different runs.
df = harness.load_all()
df = df[df["duckdb_s"].notna()].copy()
print(f"Loaded {len(df)} operations from {df['notebook'].nunique()} notebooks.")
print(C.summary())

## The headline

In [ ]:
report.headline(df)

## Results by category

The same order as the notebooks. Each block is the results table and the timing chart for one
kind of work, so you can see where the gap is widest and where it closes.

Two things to keep in mind. Anything labelled `file stats only` was answered from the Parquet
footer rather than by reading data, so it measures file opening rather than processing. And every
Spark timing carries a fixed cost of job planning and task launch that does not shrink with the
size of the question, which is why the cheapest queries show the widest gaps.

### Reading and filtering

Counting, filtering, and reading a few columns out of many.

In [ ]:
sub = df[df["category"] == "Scan"]
display(report.results_table(sub))
report.times_chart(sub, title="Reading and filtering")

### Aggregation and reporting

Group-bys, rollups, distinct counts, percentiles, top-N.

In [ ]:
sub = df[df["category"] == "Aggregation"]
display(report.results_table(sub))
report.times_chart(sub, title="Aggregation and reporting")

### Joining tables

Star-schema, big-to-big, anti-join, self-join, semi-join.

In [ ]:
sub = df[df["category"] == "Joins"]
display(report.results_table(sub))
report.times_chart(sub, title="Joining tables")

### Window functions

Ranking, running totals, rolling averages, lag and lead.

In [ ]:
sub = df[df["category"] == "Windows"]
display(report.results_table(sub))
report.times_chart(sub, title="Window functions")

### ETL and transformation

Dedup, snapshot diff, pivot, regex, JSON, business rules.

In [ ]:
sub = df[df["category"] == "ETL"]
display(report.results_table(sub))
report.times_chart(sub, title="ETL and transformation")

### Data quality checks

Null profiles, duplicates, range checks, referential integrity.

In [ ]:
sub = df[df["category"] == "Data quality"]
display(report.results_table(sub))
report.times_chart(sub, title="Data quality checks")

### Feature engineering

Feature tables, time windows, group statistics, RFM.

In [ ]:
sub = df[df["category"] == "Features"]
display(report.results_table(sub))
report.times_chart(sub, title="Feature engineering")

### Writing files out

Single file, partitioned, compaction, CSV to Parquet.

In [ ]:
sub = df[df["category"] == "Write"]
display(report.results_table(sub))
report.times_chart(sub, title="Writing files out")

## Did the answers match?

In [ ]:
ok   = int((df["same_answer"] == True).sum())
bad  = df[df["same_answer"] == False]
appr = int(df["same_answer"].isna().sum())
print(f"   Identical answers : {ok}")
print(f"   Not compared      : {appr}  (approximate algorithms)")
print(f"   Different         : {len(bad)}")
if len(bad):
    display(bad[["id", "operation", "category"]])
else:
    print("\n   Nothing differed. The only thing that varied was how long it took.")

## When distribution earns its cost

| Distribution is not earning it | Distribution earns it |
|---|---|
| The job reads gigabytes, not terabytes | The job reads terabytes |
| One job or one person at a time | Many people sharing one cluster |
| Someone is exploring, in a notebook | Two enormous tables joined together |
| A scheduled transform or quality check | Work that genuinely needs many machines |
| Tests and pipeline checks | |

## Where this pays off

Distribution is the right answer for a real class of problems, and nothing here suggests
otherwise. What has happened over the years is that "it might be big one day" turned into the
default for everything, so jobs that never needed distributing have been paying for it anyway.

These are the places where that gap is widest.

### Data science and feature engineering

Feature work is iterative. An analyst writes a window function, looks at the output, changes it,
looks again. Twenty or thirty times before anything is finished.

When each of those turns costs a cluster start plus a distributed job, people stop iterating.
They batch their questions up, guess more, and check less. The cost is not the compute bill, it
is the number of ideas that never got tested because trying one took a coffee break.

A feature pipeline that fits on one machine can run in a notebook, on a laptop, against real
data volumes, with no cluster in the loop.

### Scheduled jobs that are small but numerous

Most teams have a long tail of nightly jobs: a summary table, an extract for finance, a
reconciliation, a file drop for a partner. Individually none of them justify much thought. Added
up they hold a cluster open, and every one of them pays the start-up cost before it does a
minute of work.

These are also the safest jobs to move, because they are small, well understood, and easy to run
both ways for a fortnight before committing.

### Data quality and reconciliation

Read-only, nothing downstream breaks if they fail, and easy to verify because you can run the
same SQL both ways and compare. The lowest-risk place to start.

### Testing and CI

A test suite that needs a cluster is a test suite people skip. A test suite that runs in seconds
on the build agent is one they actually keep green. No cluster, no credentials, no queue.

### Local development against real shapes of data

Engineers can work against realistic volumes on their own machine rather than competing for a
shared environment.

## What you get with it

**It does not fall over when the data is bigger than the memory.** DuckDB streams and spills to
disk. Notebook 09 in this project processes a file several times larger than the memory it was
allowed to use, with the cap held fixed the whole way up. "Will it fit in RAM" turns out to be
the wrong question.

**There is nothing to run.** It is a library, not a service. No cluster to size, no nodes to
patch, no idle capacity, no version upgrade window. If it stops being useful, a line is deleted
from a requirements file.

**It reads what you already have.** Parquet, CSV, JSON, and Arrow directly, plus S3, Azure and
GCS through extensions, and Delta and Iceberg tables. It sits on top of the existing lake rather
than asking for the data to be moved or duplicated.

**Moving a job later is small work.** 39 of the 46 operations in this project ran on a
byte-identical SQL string in both engines. The SQL does not change and neither do the files, so
a job that outgrows one machine goes back to the cluster in an afternoon. That is the real
answer to "but what if it grows": there is very little to gain from paying for a cluster in
advance when switching costs so little.

**It is supported.** MIT licensed and open source, governed by a non-profit foundation, with
commercial support and consulting available from DuckDB Labs for teams that need a contract
behind it.

**It fits the tools people already use.** Python, pandas, Arrow and Polars all interoperate
directly, so it drops into existing notebooks and pipelines rather than replacing them.

## The point

Not that Spark should go. That "does this need distributing?" is a question worth asking per job
rather than answering once for everything, and that the default answer has a measurable price on
the jobs that never needed it.